# Document Question Answering System (RAG)

**A Retrieval-Augmented Generation pipeline for answering questions over custom documents (PDFs).**

This notebook implements the full RAG pipeline end to end:

1. Document Ingestion
2. Text Chunking
3. Embedding Creation
4. Vector Database (FAISS)
5. Query Processing
6. Context Retrieval
7. Answer Generation (Gemini API)

Instead of relying only on a language model's internal knowledge, the system retrieves relevant chunks
from your own document(s) and grounds the generated answer in that retrieved context. This improves
factual accuracy and lets you ask questions over private / domain-specific data.


## 1. Setup

Install the required libraries. Run this cell once.

- `pypdf` — read text out of PDF files
- `sentence-transformers` — turn text chunks into embedding vectors (runs locally, no API key needed)
- `faiss-cpu` — fast vector similarity search (our vector database)
- `google-generativeai` — call the Gemini API for the final answer generation step


In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu google-generativeai

In [ ]:
import os
import textwrap
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import google.generativeai as genai


## 2. Configuration

Set your **PDF path** and your **Gemini API key** below.

Get a free Gemini API key here: https://aistudio.google.com/app/apikey

> Tip: don't hardcode your real key if you're going to share this notebook — you can also set it via
> an environment variable: `os.environ["GEMINI_API_KEY"]`.


In [ ]:
PDF_PATH = "your_document.pdf"          # <-- put your PDF file path here
GEMINI_API_KEY = "PASTE_YOUR_API_KEY_HERE"  # <-- put your Gemini API key here

CHUNK_SIZE = 500      # characters per chunk
CHUNK_OVERLAP = 50    # overlap between consecutive chunks, helps preserve context across chunk boundaries
TOP_K = 3             # number of chunks to retrieve per query

genai.configure(api_key=GEMINI_API_KEY)
generation_model = genai.GenerativeModel("gemini-2.5-flash")


## 3. Document Ingestion

Load the PDF and extract raw text from every page.


In [ ]:
def load_pdf(path: str) -> str:
    """Extract and concatenate text from every page of a PDF."""
    reader = PdfReader(path)
    full_text = ""
    for page in reader.pages:
        page_text = page.extract_text() or ""
        full_text += page_text + "\n"
    return full_text


## 4. Text Chunking

Long documents are split into smaller overlapping chunks. Smaller chunks improve retrieval accuracy
because the embedding for a short, focused chunk captures its meaning more precisely than the embedding
of an entire document would.


In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks of `chunk_size` characters."""
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return [c for c in chunks if c.strip()]


## 5. Embedding Creation

Each chunk is converted into a dense vector that captures its semantic meaning, using a local
`sentence-transformers` model (`all-MiniLM-L6-v2`) — this runs on your machine, no API call needed.


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_chunks(chunks: list[str]) -> np.ndarray:
    """Convert a list of text chunks into an array of embedding vectors."""
    embeddings = embedding_model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)
    return embeddings.astype("float32")


## 6. Vector Database (FAISS)

Store the chunk embeddings in a FAISS index for fast similarity search. We use `IndexFlatL2`, which
does an exact nearest-neighbour search — simple and accurate, ideal for smaller documents.


In [ ]:
def build_vector_store(embeddings: np.ndarray) -> faiss.IndexFlatL2:
    """Build a FAISS index from an array of embeddings."""
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index


## 7. Query Processing & Context Retrieval

Convert the user's question into an embedding using the *same* embedding model, then search the FAISS
index for the `TOP_K` most similar chunks.


In [ ]:
def retrieve_relevant_chunks(query: str, index: faiss.IndexFlatL2, chunks: list[str], top_k: int = TOP_K) -> list[str]:
    """Embed the query and retrieve the top_k most similar chunks from the vector store."""
    query_embedding = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return [chunks[i] for i in indices[0] if i != -1]


## 8. Answer Generation

Combine the retrieved chunks into a context block and pass it to the Gemini model along with the
user's question, so the final answer is grounded in the actual document content rather than the
model's general knowledge.


In [ ]:
def generate_answer(query: str, context_chunks: list[str]) -> str:
    """Generate a grounded answer using the retrieved context."""
    context = "\n\n---\n\n".join(context_chunks)
    prompt = f"""You are a helpful assistant answering questions using ONLY the context provided below.
If the answer isn't in the context, say you don't have enough information from the document.

Context:
{context}

Question: {query}

Answer:"""
    response = generation_model.generate_content(prompt)
    return response.text


## 9. Full RAG Pipeline

This ties every stage together: ingest the PDF once, then answer as many questions as you like against it.


In [ ]:
class RAGPipeline:
    def __init__(self, pdf_path: str):
        print("Loading and chunking document...")
        raw_text = load_pdf(pdf_path)
        self.chunks = chunk_text(raw_text)
        print(f"Created {len(self.chunks)} chunks.")

        print("Creating embeddings...")
        embeddings = embed_chunks(self.chunks)

        print("Building vector store...")
        self.index = build_vector_store(embeddings)
        print("Ready! Ask a question with .ask(\"your question\")")

    def ask(self, query: str) -> str:
        relevant_chunks = retrieve_relevant_chunks(query, self.index, self.chunks)
        answer = generate_answer(query, relevant_chunks)
        return answer


## 10. Run It

Build the pipeline on your PDF, then ask questions.


In [ ]:
rag = RAGPipeline(PDF_PATH)


In [ ]:
question = "What is the main idea of the document?"
answer = rag.ask(question)
print("Q:", question)
print("A:", answer)


In [ ]:
# Try your own question
question = "Type your own question here"
answer = rag.ask(question)
print("Q:", question)
print("A:", answer)


## 11. Key Learnings

- **Retrieval** finds the most relevant chunks of text using embeddings and vector similarity search.
- **Augmentation** injects retrieved chunks into the model's input as grounding context.
- **Generation** produces the final answer, constrained to the retrieved information.
- Chunk size and overlap directly affect retrieval quality — too large loses precision, too small loses context.
- Vector databases (here, FAISS) enable fast similarity search even over large document collections.

## 12. Possible Improvements

- Hybrid search (keyword + vector) for better retrieval
- Add a re-ranking step over the retrieved chunks
- Try different embedding models (e.g. `all-mpnet-base-v2` for higher accuracy)
- Support multiple documents / a persistent vector store (e.g. Pinecone, Chroma)
- Add a simple UI (Streamlit/Gradio) for interactive Q&A

## Conclusion

This notebook demonstrates a complete Retrieval-Augmented Generation system: it ingests a custom PDF,
chunks and embeds its content, stores it in a vector database, retrieves the most relevant context for
a user's question, and generates a grounded answer using the Gemini API. This is the same pattern used
in real-world chatbots, knowledge assistants, enterprise search, and AI documentation tools.
